In [ ]:
import os
import json
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# === Paths ===
train_json_path = "../RadSpineXR/Train/llava_dataset/train/dataset.json"
val_json_path = "../RadSpineXR/Train/llava_dataset/validation/dataset.json"
image_folder = "../RadSpineXR/Train/llava_dataset/images"
output_dir = "./qwen2.5-vl-finetuned3-spine"

# === Visual Token Bounds (optional)
min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28

# === Load Processor
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    min_pixels=min_pixels,
    max_pixels=max_pixels,
    use_fast=False
)

# === Load Model
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

# === Load Datasets
with open(train_json_path, 'r') as f:
    train_data = json.load(f)

with open(val_json_path, 'r') as f:
    val_data = json.load(f)

# === Dataset Class Supporting 1 or 6 QA Pairs per Image
class SpineXRQwenDataset(Dataset):
    def __init__(self, data, image_folder, processor):
        self.samples = []
        self.processor = processor
        self.image_folder = image_folder

        for item in data:
            image_path = os.path.join(image_folder, item["image"])
            conversations = item["conversations"]
            for i in range(0, len(conversations), 2):
                q = conversations[i]["value"]
                a = conversations[i + 1]["value"]
                self.samples.append({
                    "image_path": image_path,
                    "question": q,
                    "answer": a
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["image_path"]).convert("RGB")
        question = sample["question"]
        answer = sample["answer"]

        # Prepare prompt: only image + question
        prompt_only = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image,
                        "resized_height": 392,
                        "resized_width": 392
                    },
                    {
                        "type": "text",
                        "text": question
                    }
                ]
            }
        ]
        full_conversation = [
            *prompt_only,
            {"role": "assistant", "content": answer}
        ]

        prompt_text = self.processor.apply_chat_template(prompt_only, tokenize=False, add_generation_prompt=True)
        full_text = self.processor.apply_chat_template(full_conversation, tokenize=False, add_generation_prompt=False)

        image_inputs, video_inputs = process_vision_info(full_conversation)
        model_inputs = self.processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        model_inputs = {k: v.squeeze(0) for k, v in model_inputs.items()}

        # Correct way to align labels
        prompt_ids = self.processor.tokenizer(
            prompt_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )["input_ids"].squeeze(0)

        labels = model_inputs["input_ids"].clone()
        labels[:len(prompt_ids)] = -100  # mask prompt
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        model_inputs["labels"] = labels

        return model_inputs

# === Create Datasets
train_dataset = SpineXRQwenDataset(train_data, image_folder, processor)
val_dataset = SpineXRQwenDataset(val_data, image_folder, processor)

# === Collator
def collate_fn(batch):
    return {
        key: torch.nn.utils.rnn.pad_sequence(
            [x[key] for x in batch],
            batch_first=True,
            padding_value=(processor.tokenizer.pad_token_id if key != "labels" else -100)
        )
        for key in batch[0]
    }

# === Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    save_total_limit=2,
    learning_rate=4e-5,
    warmup_ratio=0.1,
    weight_decay=0.05,
    logging_dir="./logs_qwen",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=[],
    fp16=False,
    bf16=True,
    dataloader_num_workers=2,
)

# === Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    tokenizer=processor.tokenizer,
)

# === Train
print("\U0001F680 Starting Qwen2.5-VL fine-tuning...")
trainer.train()
print("\u2705 Training complete.")

# === Save Model
trainer.save_model(output_dir)
print(f"\U0001F4BE Fine-tuned Qwen2.5-VL model saved at: {output_dir}")

In [ ]:

import matplotlib.pyplot as plt

# Data
steps = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600]
training_loss = [0.164200, 0.100400, 0.081600, 0.055900, 0.050300, 0.056100, 0.032200,
                 0.035200, 0.031200, 0.026400, 0.027600, 0.025700, 0.025400, 0.021600, 0.017200, 0.019100]
validation_loss = [0.156943, 0.100539, 0.084378, 0.078427, 0.074433, 0.071773, 0.076361,
                   0.073912, 0.073812, 0.073855, 0.080144, 0.081865, 0.082686, 0.087972, 0.089153, 0.089535]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(steps, training_loss, label='Training Loss', marker='o')
plt.plot(steps, validation_loss, label='Validation Loss', marker='s')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import os
import json
from pathlib import Path

import torch
from PIL import Image
import pandas as pd
from tqdm import tqdm

from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# =========================
# CONFIG: EDIT THESE
# =========================
TEST_JSON_PATH = "../RadSpineXR/Test/sample_150.json"   # your test json
IMAGE_FOLDER   = "../RadSpineXR/Test/Unannoated_images_150"  # folder with images
OUT_CSV        = "./qwen2_5_vl_zeroshot_spine.csv"

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.0
TOP_P          = 1.0

# Optional visual token bounds (same as your finetune script)
min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28

# =========================
# DEVICE SETUP
# =========================
if torch.cuda.is_available():
    DEVICE = "cuda:1"   # or "cuda:0" depending on what you want
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

# dtype for model
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# =========================
# LOAD PROCESSOR & MODEL
# =========================
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
    use_fast=False,
)

# ❗ Removed device_map="auto" so accelerate is NOT required
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,   # you can also use dtype=DTYPE in newer versions
)

# Move entire model to the chosen device
model.to(DEVICE)
model.eval()

# Use the same device variable later
device = DEVICE

# Make sure we have a pad token
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

pad_token_id = processor.tokenizer.pad_token_id

# =========================
# LOAD TEST JSON
# =========================
TEST_JSON_PATH = Path(TEST_JSON_PATH)
with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} image entries from {TEST_JSON_PATH}")

# =========================
# ZERO-SHOT INFERENCE
# =========================
rows = []

for item in tqdm(test_data, desc="Running zero-shot inference"):
    image_rel_path = item["image"]
    image_path = os.path.join(IMAGE_FOLDER, image_rel_path)

    if not os.path.exists(image_path):
        print(f"[WARN] Image not found: {image_path}")
        continue

    conversations = item["conversations"]

    # Expecting pairs: user (question), assistant (answer)
    # (If your JSON is not perfectly paired, you might want a safety check here)
    for i in range(0, len(conversations), 2):
        # safety: skip if incomplete pair
        if i + 1 >= len(conversations):
            break

        user_turn = conversations[i]
        assistant_turn = conversations[i + 1]

        question = user_turn["value"]
        ground_truth = assistant_turn["value"]

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Build messages for zero-shot (only image + question)
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image,
                        "resized_height": 392,
                        "resized_width": 392,
                    },
                    {
                        "type": "text",
                        "text": question,
                    },
                ],
            }
        ]

        # Apply Qwen chat template
        prompt_text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,  # so model knows it should answer
        )

        # Directly pass the PIL image (no qwen_vl_utils)
        image_inputs = [image]

        # Tokenize & prepare inputs
        inputs = processor(
            text=[prompt_text],
            images=image_inputs,
            return_tensors="pt",
        )

        # Move to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Generate
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=(TEMPERATURE > 0),
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=pad_token_id,
            )

        # Remove prompt tokens from output
        gen_ids = generated_ids[:, inputs["input_ids"].shape[1]:]

        # Decode prediction
        prediction = processor.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )[0].strip()

        rows.append({
            "image": image_rel_path,
            "question": question,
            "ground_truth": ground_truth,
            "prediction": prediction,
        })

# =========================
# SAVE TO CSV
# =========================
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print(f"\n✅ Zero-shot predictions saved to: {OUT_CSV}")
print(f"Total QA pairs processed: {len(df)}")
